# 05. Prospective Linebuster Business Case & ROI Scenario Simulator

**Ex-Ante Modeling (`LINEBUSTER_START_DATE = None`):** The linebuster has **not** been deployed yet. This notebook:
1. Profiles peak-hour congestion windows and empirical P90/P95 throughput ceilings (`Hourly_Sales`).
2. Normalizes baseline demand by removing Silao OSINT external-incident days.
3. Audits `2026-09-05` (`Tablet` / `Contextual_Split`) strictly as a `pos_channel_change` segmentation check (not a linebuster launch).
4. Simulates Conservative / Base / Optimistic peak-hour lift scenarios, break-even tickets/day, monthly net profit, payback period, and margin sensitivity.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path(os.environ.get("BGA_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_ROOT / "src").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

os.environ.setdefault("MPLCONFIGDIR", "/tmp/bga-matplotlib")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

from src.causal_eval import (
    analyze_attach_and_segmentation,
    build_osint_normalized_baseline,
    build_sensitivity_matrix,
    simulate_linebuster_scenarios,
)
from src.config import load_config
from src.osint_collector import load_incidents_file

DATA_MODE = os.environ.get("BGA_DATA_MODE", "sample").strip().lower()
cfg = load_config(repo_root=REPO_ROOT)
print(
    f"DATA_MODE={DATA_MODE} | linebuster_start_date={cfg.linebuster_start_date} | "
    f"tablet_interpretation={cfg.tablet_channel_interpretation}"
)

In [ ]:
if DATA_MODE == "raw":
    from src.db_loader import (
        load_contextual_split,
        load_daily_sales,
        load_hourly_sales,
        load_product_mix,
        load_sales_channels,
    )
    daily_df = load_daily_sales(config=cfg)
    hourly_df = load_hourly_sales(config=cfg, complete_grid=True)
    mix_df = load_product_mix(config=cfg)
    ctx_df = load_contextual_split(config=cfg)
    ch_df = load_sales_channels(config=cfg)
else:
    daily_df = pd.read_csv(cfg.sample_dir / "daily_sales.csv", parse_dates=["date"])
    hourly_df = pd.read_csv(cfg.sample_dir / "hourly_sales.csv", parse_dates=["date"])
    mix_df = pd.read_csv(cfg.sample_dir / "product_mix.csv", parse_dates=["date"])
    ctx_df = None
    ch_df = None

incidents_df = load_incidents_file(cfg.sample_dir / "incidents.csv", relevant_only=True)
baseline = build_osint_normalized_baseline(
    daily_df=daily_df,
    hourly_df=hourly_df,
    incidents_df=incidents_df,
    store_lat=cfg.lat,
    store_lon=cfg.lon,
    max_distance_m=1500.0,
    peak_hours=cfg.peak_hours,
    operating_hours=cfg.operating_hours,
)
seg = analyze_attach_and_segmentation(
    daily_df=daily_df,
    product_mix_df=mix_df,
    contextual_split_df=ctx_df,
    sales_channels_df=ch_df,
    tablet_start_date=cfg.tablet_channel_start_date,
    tablet_interpretation=cfg.tablet_channel_interpretation,
    items_end_date=cfg.items_coverage_end_date,
)
print(
    f"Clean Days={baseline['clean_days']}/{baseline['total_days']} | "
    f"Clean Daily Rev={baseline['clean_mean_daily_revenue']:.2f} | "
    f"Peak Share={baseline['peak_revenue_share']:.1%} | "
    f"Peak P95 Ceiling={baseline['peak_p95_daily_ceiling']:.2f}"
)
print(f"Food Revenue Share={seg['food_revenue_share']:.1%} | Pre-Aug20 Combo Unit Share={seg['combo_unit_share_pre_cutoff']:.1%}")

In [ ]:
scenarios_df = simulate_linebuster_scenarios(baseline=baseline, config=cfg)
print("Prospective 3-Scenario Linebuster Business Case:")
print(
    scenarios_df[
        [
            "scenario",
            "peak_lift_pct",
            "attach_lift_pct",
            "total_incremental_rev_day",
            "monthly_net_incremental_profit",
            "breakeven_tickets_per_day",
            "payback_months",
            "pct_of_p95_peak_ceiling",
        ]
    ].to_string(index=False)
)

sens_df = build_sensitivity_matrix(baseline=baseline, config=cfg)
pivot = sens_df.pivot(index="gross_margin_pct", columns="peak_lift_pct", values="monthly_net_incremental_profit")
print("\nMonthly Net Incremental Profit Sensitivity (Gross Margin x Peak Lift):")
print(pivot.round(1).to_string())

htbl = baseline["clean_bottleneck"]["hourly_table"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(htbl["hour_int"], htbl["mean_volume"], marker="o", label="Mean Hourly Rev")
axes[0].plot(htbl["hour_int"], htbl["p95_volume"], linestyle="--", marker="^", label="P95 Capacity Ceiling")
axes[0].set_title("Hourly Demand vs Empirical P95 Ceiling")
axes[0].legend()

axes[1].bar(scenarios_df["scenario"], scenarios_df["monthly_net_incremental_profit"], color=["#66c2a5", "#3288bd", "#5e4fa2"])
axes[1].axhline(0.0, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title(f"Projected Monthly Net Profit ({cfg.currency})")
plt.tight_layout()
plt.close(fig)